In [ ]:
import json
import os

from common.carp_setup.fch_setup import simulation
# from common.carp_setup.cell_sims_utils import generate_bench_script

In [ ]:
# Probably won't change

username                        = "jsolisle"
platform = "archer2"

# Particular experiment

folder_experiment_name = "HCM/4/scenarios/49_more_samples"
num_beats_limit_cycle  = 500
# num_beats              = 5

# HPC
# /work/e348/e348/<archer2_username>/rodero_healthy/h11/scenarios/5/states
# states_folder = f"/rds/general/user/{username}/home/{folder_experiment_name}/states" # put here the .sv files on the hpc
harddrive = "/scratch-nvme/e348/e348"
states_folder = f"{harddrive}/{username}/{folder_experiment_name}/states" # put here the .sv files on the hpc

# Local computer
# base_folder_local   = f"/path/to/data/SeagateExpansionDrive/{folder_experiment_name}"
local_harddrive="/path/to/data/Elements/"
base_folder_local   = f"/{local_harddrive}/{folder_experiment_name}"
cell_sims_basefolder = f"{base_folder_local}/cell_sims"
torord_folder       = f"{base_folder_local}/SS/param"
courtemanche_folder = f"{base_folder_local}/SS/param"
data_folder          = f"{base_folder_local}/data"
json_folder = f"{base_folder_local}/json_files"

os.makedirs(cell_sims_basefolder, exist_ok=True)

sim_setup = simulation()
sim_setup.load(f"{json_folder}/{platform}_setup.json")

with open(f"{base_folder_local}/json_files/clinical_data.json","r") as f:
    clinical_data = json.load(f)


first_simulation = 800
last_simulation = 2000

In [ ]:
def generate_bench_script_flexible(first_simulation,
						  last_simulation,
						  BCL,
						  NBEATS,
						  basefolder,
						  NPROC,
						  strain,
						  chamber,
						  contraction_model,
						  suffix):

	if basefolder[-1]=="/":
		basefolder = basefolder[:-1]
	
	if chamber=="LV":
		ionic_model = "ToRORd_dynCl"
	elif chamber=="RV":
		ionic_model = "ToRORd_dynCl"
	elif chamber=="atria":
		ionic_model = "JB_COURTEMANCHE"
	else:
		raise Exception("Unsupported chamber - pick LV, RV or atria")

	f = open(basefolder+"/run_"+ionic_model+suffix+".sh","w")

	f.write("#!/bin/bash\n")

	os.system("mkdir "+basefolder+"/"+ionic_model+suffix+"/")

	f.write("\n")
	
	DURATION  = NBEATS*BCL
	N_start1 = first_simulation
	N_end1    = ((last_simulation+1)//NPROC)*NPROC-1
	N_start2  = ((last_simulation+1)//NPROC)*NPROC	
	N_end2    = last_simulation

	cmd=["cmd="+'"'+"bench.pt"]
	cmd += ["--numstim",str(NBEATS)+"\n"]
	cmd += ["--bcl",str(BCL)+"\n"]
	cmd += ["--past-stim",str(BCL)+"\n"]
	cmd += ["--imp",ionic_model+"\n"]
	cmd += ["--dt","0.02"+"\n"]
	cmd += ["--stim-curr","60.0"+"\n"]
	cmd += ["--dt-out","1.0"+"\n"]

	if contraction_model=="Land":
		cmd += ["--plug-in","LandHumanStress"+"\n"]
		cmd += ["--save-ini-file","${output}/${PBS_ARRAY_INDEX}_"+ionic_model+suffix+"_LandHumanStress.sv"+"\n"]
	elif contraction_model=="tanh":
		cmd += ["--plug-in","TanhStress"+"\n"]
		cmd += ["--save-ini-file","${output}/${PBS_ARRAY_INDEX}_"+ionic_model+suffix+"_TanhStress"+suffix+".sv"+"\n"]

	cmd += ["--strain",str(strain)+"\n"]
	cmd += ["--strain-rate","0.0"+"\n"]
	cmd += ["--strain-time","0.0"+"\n"]
	cmd += ["--strain-dur",str(DURATION)+"\n"]
	cmd += ["--imp-par","${pp_ionic}"+"\n"]
	cmd += ["--plug-par","${pp_land}"+"\n"]
	cmd += ["--save-ini-time ",str(DURATION)+"\n"]

	cmd += ["--fout=${output}/"+ionic_model+"\n"]
	cmd += ["--plug-sv-dump=Tension"+"\n"]
	cmd += ["--imp-sv-dump=Ca_i"+"\n"]

	cmd_str = " ".join(cmd)
	cmd_final = cmd_str+" --bin\n --no-trace\n >/dev/null 2>&1 &"+'"\n'

	# --------------------------------------------------
	# first loop 
	if N_end1>=0:
		f.write("NPROC="+str(NPROC)+"\n")
		f.write("i=0\n")

		f.write("\n")
		f.write("\n")

		f.write(f"for PBS_ARRAY_INDEX in $(seq {N_start1} "+str(N_end1)+" )\n")	

		f.write("do\n")
		f.write("\n")	

		f.write("pp_ionic=$(cat "+basefolder+"/param/${PBS_ARRAY_INDEX}_param_"+ionic_model+".txt)\n")
		f.write("pp_land=$(cat "+basefolder+"/param/${PBS_ARRAY_INDEX}_param_"+ionic_model+"_"+contraction_model+suffix+".txt)\n")	

		f.write("\n")	

		f.write("output="+basefolder+"/"+ionic_model+suffix+"/$PBS_ARRAY_INDEX\n")
		f.write("mkdir ${output}\n")	

		f.write("\n")
		f.write(cmd_final)
		f.write("\n")
		f.write("eval $cmd\n")
		f.write("\n")

		f.write("((i++))\n")
		f.write("[[ $((i%NPROC)) -eq 0 ]] && wait\n")
		f.write("done\n")

		f.write("\n")
		f.write("\n")
		f.write("\n")
		f.write("\n")

	f.write("NPROC="+str((last_simulation+1)-N_start2)+"\n")
	f.write("i=0\n")
	f.write("\n")
	f.write("\n")

	f.write("for PBS_ARRAY_INDEX in $(seq "+str(N_start2)+" "+str(N_end2)+" )\n")	

	f.write("do\n")
	f.write("\n")	

	f.write("pp_ionic=$(cat "+basefolder+"/param/${PBS_ARRAY_INDEX}_param_"+ionic_model+".txt)\n")
	f.write("pp_land=$(cat "+basefolder+"/param/${PBS_ARRAY_INDEX}_param_"+ionic_model+"_"+contraction_model+suffix+".txt)\n")	

	f.write("\n")	

	f.write("output="+basefolder+"/"+ionic_model+suffix+"/$PBS_ARRAY_INDEX\n")
	f.write("mkdir ${output}\n")	

	f.write("\n")
	f.write(cmd_final)
	f.write("eval $cmd\n")
	f.write("\n")
	f.write("\n")
	f.write("((i++))\n")
	f.write("[[ $((i%NPROC)) -eq 0 ]] && wait\n")
	f.write("done\n")	


In [ ]:
generate_bench_script_flexible(first_simulation=first_simulation,
							   last_simulation=last_simulation,
                      BCL=clinical_data["general"]["BCL"],                                # BCL
                      NBEATS=num_beats_limit_cycle,                            # NBEATS
                      basefolder=f"{base_folder_local}/SS/",                # basefolder containing the param folder
                      NPROC=23,                            # number of CPUs to use for parallel runs
                      strain=0.0,                                # strain
                      chamber="LV",                                 # chamber = LV, RV or atria
                      contraction_model=sim_setup.contraction_model,         # contraction model
                      suffix="")

os.system(f"bash {base_folder_local}/SS/run_ToRORd_dynCl.sh")


In [ ]:
generate_bench_script_flexible(first_simulation=first_simulation,
							   last_simulation=last_simulation,
                      BCL=clinical_data["general"]["BCL"],                                # BCL
                      NBEATS=num_beats_limit_cycle,                            # NBEATS
                      basefolder=f"{base_folder_local}/SS/",                # basefolder containing the param folder
                      NPROC=23,                            # number of CPUs to use for parallel runs
                      strain=0.0,                                # strain
                      chamber="RV",                                 # chamber = LV, RV or atria
                      contraction_model=sim_setup.contraction_model,         # contraction model
                      suffix="_rv")

os.system(f"bash {base_folder_local}/SS/run_ToRORd_dynCl_rv.sh")


# Atria: Converted COURTEMANCHE + LandHumanStress

In [ ]:
generate_bench_script_flexible(first_simulation=first_simulation,
							   last_simulation=last_simulation,
                      BCL=clinical_data["general"]["BCL"],                                # BCL
                      NBEATS=num_beats_limit_cycle,                            # NBEATS
                      basefolder=f"{base_folder_local}/SS/",                # basefolder containing the param folder
                      NPROC=23,                            # number of CPUs to use for parallel runs
                      strain=0.0,                                # strain
                      chamber="atria",                                 # chamber = LV, RV or atria
                      contraction_model=sim_setup.contraction_model,         # contraction model
                      suffix="")

os.system(f"bash {base_folder_local}/SS/run_JB_COURTEMANCHE.sh")


# Quality check

In [ ]:
def plot_Land_output(basefolder,
                     N,
                     figname=None,
                     isometric=False,
                     figsize=(10, 5),
                     mask=[],
                     default='',
                     color='#3489eb'):

    """
    Plot Land output calcium and tension.

    Args:
        - basefolder: containing all simulations, folder numbered from
                    0 to N-1
        - N: number of simulations to plot
        - figname: path to output figure. If None, the plot is shown
        - isometric: if True, no stretch is plotted
        - figsize: tuple, (width, height)
        - mask: boolean mask containing which simulations terminated successfully
                and which ones didn't
        - default: path to folder with Land output (Tension.dat for instance) 
                   that can be plotted against the simulations for comparison

    """

    if not isometric:
        ax = plt.figure(figsize=figsize, constrained_layout=True).subplots(1, 3)
    else:
        ax = plt.figure(figsize=figsize, constrained_layout=True).subplots(1, 2)

    plot_all = np.arange(N)
    if len(mask) > 0:
        plot_idx = plot_all[np.where(mask == 1)[0]]
    else:
        plot_idx = plot_all

    for i in plot_all:
        if not os.path.exists(basefolder+'/'+str(i)+'/Tension.dat'):
            raise Exception('Cannot find output file. The folder structure needs to be basefolder/i/Tension.dat and Ca_i.dat')

        T = cell_io.read_ionic_output(basefolder+'/'+str(i)+'/Tension.dat')
        Ca_i = cell_io.read_ionic_output(basefolder+'/'+str(i)+'/Ca_i.dat')
        t = np.arange(0, Ca_i.shape[0])

        if i in plot_idx:
            if np.max(np.abs(T)) < 500.0:
                # Plot the curve
                ax[0].plot(t, Ca_i, color=color)

                # Add a red star at the first point of the curve
                ax[0].plot(t[0], Ca_i[0], marker='*', markersize=8, color='red')

                ax[1].plot(t, T, color=color)
                ax[1].plot(t[0], T[0], marker='*', markersize=8, color='red')
        else:
            if np.max(np.abs(T)) < 500.0:
                ax[0].plot(t, Ca_i, color='black', zorder=0)
                ax[1].plot(t, T, color='black', zorder=0)

        ax[0].set_xlabel('Time [ms]')
        ax[1].set_xlabel('Time [ms]')

        ax[0].set_ylabel('Ca_i [um]')
        ax[1].set_ylabel('Tension [kPa]')

        # lambda_out = read_ionic_output(basefolder+'/'+str(i)+'/lambda.dat')
        if not isometric:
            lambda_out = cell_io.read_ionic_output(basefolder+'/'+str(i)+'/stretch.dat')
            if i in plot_idx:
                ax[2].plot(t, lambda_out, color=color)
                ax[2].plot(t[0], lambda_out[0], marker='*', markersize=8, color='red')
            else:
                ax[2].plot(t, lambda_out, color='black', zorder=0)

            ax[2].set_xlabel('Time [ms]')
            ax[2].set_ylabel('Lambda [-]')

        if default != '':
            T = cell_io.read_ionic_output(default+'/Tension.dat')
            Ca_i = cell_io.read_ionic_output(default+'/Ca_i.dat')

            ax[0].plot(t, Ca_i, '--',
                       color='black',
                       linewidth=2.0
                       )
            ax[1].plot(t, T, '--',
                       color='black',
                       linewidth=2.0
                       )

    if figname is None:
        plt.show()
    else:
        plt.savefig(figname, dpi=100)

In [ ]:
from common import cell_io
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
ionic_models = ["ToRORd_dynCl","ToRORd_dynCl","JB_COURTEMANCHE"]
# ionic_models = ["JB_COURTEMANCHE"]
suffix = ["","_rv",""]
# suffix = [""]
for i in range(len(ionic_models)):
    cell_io.bin_to_dat_folder(sim_foldername = f"{base_folder_local}/SS/{ionic_models[i]}{suffix[i]}/",
                          start_sample=0,
                          last_sample=799,
                          BCL=int(clinical_data["general"]["BCL"]),
                          Nbeats=int(num_beats_limit_cycle),
                          bin_files=[f"{ionic_models[i]}.Vm.bin",f"{ionic_models[i]}.Ca_i.bin",f"{ionic_models[i]}.Tension.bin"],
                          output_files=[f"{ionic_models[i]}.Vm.dat","Ca_i.dat","Tension.dat"],
                          cleanup=False)

    plot_Land_output(basefolder =  f"{base_folder_local}/SS/{ionic_models[i]}{suffix[i]}/",
                          N = 800,
                          isometric = True,
                          figname=f"{base_folder_local}/SS/{ionic_models[i]}{suffix[i]}_Land_output.png")

In [ ]:

basefolder_1 = "/path/to/data/SeagateExpansionDrive/rodero_healthy/h11/scenarios/alex/cell_sims"
basefolder_2 = "/path/to/data/SeagateExpansionDrive/rodero_healthy/h11/scenarios/10/SS/ToRORd_dynCl/89"
BCL=1000
Nbeats = 5

bin_files = [f"ToRORd_dynCl.Vm.bin",f"ToRORd_dynCl.Ca_i.bin",f"ToRORd_dynCl.Tension.bin"]
output_files = [f"ToRORd_dynCl.Vm.dat","Land.Ca_i.dat","Land.Tension.dat"]


for j,filename in enumerate(bin_files):

    full_filename = f"{basefolder_1}/{filename}"
    cell_io.bin_to_dat(full_filename,
                BCL,
                Nbeats,
                f"{basefolder_1}/{output_files[j]}",
                cleanup=False
                )



ax = plt.figure(figsize=(10,5), constrained_layout=True).subplots(1, 3)


T_1 = cell_io.read_ionic_output(f"{basefolder_1}/{output_files[2]}")
Ca_i_1 = cell_io.read_ionic_output(f"{basefolder_1}/{output_files[1]}")
Vm_1 = cell_io.read_ionic_output(f"{basefolder_1}/{output_files[0]}")
t_1 = np.arange(0, Ca_i_1.shape[0])

T_2 = cell_io.read_ionic_output(f"{basefolder_2}/{output_files[2]}")
Ca_i_2 = cell_io.read_ionic_output(f"{basefolder_2}/{output_files[1]}")
Vm_2 = cell_io.read_ionic_output(f"{basefolder_2}/{output_files[0]}")
t_2 = np.arange(0, Ca_i_2.shape[0])

if np.max(np.abs(T_1)) < 500.0:
        # Plot the curve
        ax[0].plot(t_1, Ca_i_1, color='#3489eb')
        ax[0].plot(t_2, Ca_i_2, color='red')

        # Add a red star at the first point of the curve
        ax[0].plot(t_1[0], Ca_i_1[0], marker='*', markersize=8, color='red')

        ax[1].plot(t_1, T_1, color='#3489eb')
        ax[1].plot(t_2, T_2, color='red')
        ax[1].plot(t_1[0], T_1[0], marker='*', markersize=8, color='blue')
        ax[1].plot(t_2[0], T_2[0], marker='*', markersize=8, color='red')


        ax[2].plot(t_1, Ca_i_1, color='#3489eb')
        ax[2].plot(t_2, Ca_i_2, color='red')
else:
    if np.max(np.abs(T_1)) < 500.0:
        ax[0].plot(t_1, Ca_i_1, color='black', zorder=0)
        ax[1].plot(t_1, T_1, color='black', zorder=0)

ax[0].set_xlabel('Time [ms]')
ax[1].set_xlabel('Time [ms]')

ax[0].set_ylabel('Ca_i [um]')
ax[1].set_ylabel('Tension [kPa]')
ax[2].set_ylabel('Vm')


plt.show()